In [1]:
import pandas as pd
import re

# STEP 1: Define all partition files
files = [
    "alfar_part1.json",
    "alfar_part2.json",
    "alfar_part3.json",
    "alfar_part4.json"
]

# This will store processed data from each file
df_list = []


# STEP 2: Function to extract audit fields (date and newQty)
def extract_audit_fields(audits):
    result = {}

    # Check if audit column contains a list
    if isinstance(audits, list):
        for i, audit in enumerate(audits, start=1):

            # Ensure each audit entry is a dictionary
            if isinstance(audit, dict):

                # Extract audit date
                result[f"audit{i}_date"] = audit.get("auditDate", {}).get("$date")

                # Extract updated quantity
                result[f"audit{i}_newQty"] = audit.get("newQty")

        # STEP 3: Get the latest quantity (highest audit index)
        qty_keys = [k for k in result if "newQty" in k]

        if qty_keys:
            last_key = sorted(
                qty_keys,
                key=lambda x: int(re.findall(r"\d+", x)[0])
            )[-1]

            result["latestQty"] = result[last_key]

    return pd.Series(result)


# STEP 4: Function to extract the latest audit date
def get_latest_audit_date(audits):
    if isinstance(audits, list) and len(audits) > 0:

        last_audit = audits[-1]  # last record = most recent

        if isinstance(last_audit, dict):
            return last_audit.get("auditDate", {}).get("$date")

    return None


# STEP 5: Process each partition file
for file in files:
    print("Processing:", file)

    # Load JSON file
    data = pd.read_json(file)

    # Extract audit fields (dates and quantities)
    audit_cols = data["audit"].apply(extract_audit_fields)

    # Extract latest audit date
    data["latestAuditDate"] = data["audit"].apply(get_latest_audit_date)

    # Merge extracted audit fields back into dataset
    data = pd.concat([data, audit_cols], axis=1)

    # STEP 6: Keep only important columns
    data = data[
        [
            "_id",
            "productName",
            "qty",
            "branch",
            "sku",
            "category",
            "type",
            "latestQty",
            "latestAuditDate"
        ]
    ]

    # Store processed partition
    df_list.append(data)


# STEP 7: Combine all partitions into one dataset
df_final = pd.concat(df_list, ignore_index=True)


# STEP 8: Create audit priority based on stock quantity
def assign_priority(qty):
    if qty < 0:
        return "High"
    elif qty == 0:
        return "Medium"
    else:
        return "Low"

df_final["audit_priority"] = df_final["qty"].apply(assign_priority)


# STEP 9: Save final dataset to CSV
df_final.to_csv("alfar_priority_audit_list_updated.csv", index=False)

print("Finished! Total rows:", df_final.shape)

Processing: alfar_part1.json


KeyboardInterrupt: 